# Convert citation_network.gml → Edge_list.parquet

Parses `citation_network.gml` (~2.6GB) in a streaming fashion and saves the edge list as Parquet.

**File structure** (standard GML written by networkx):
```
graph [
  directed 1
  node [
    id 0          ← internal integer id
    label "1"     ← actual node name (case/document ID)
  ]
  ...
  edge [
    source 0      ← refers to internal id
    target 1
  ]
]
```

Because the file is very large, we use a **line-by-line streaming parser** instead of `networkx.read_gml()`.
(The full graph is never loaded into memory — only the node id → label mapping is kept.)

**Output**: `Edge_list.parquet` — two columns `source`, `target` (original label values, int64)

In [ ]:
from pathlib import Path
import time

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

GML_PATH = Path(r"c:\Users\jdwoo\OneDrive\Desktop\Research\caselaw\caselaw\citation_network.gml")
OUT_PATH = Path(r"c:\Users\jdwoo\OneDrive\Desktop\Research\caselaw\caselaw\Edge_list.parquet")

# True  → store source/target as the original labels (case IDs) — recommended
# False → store the GML internal integer ids (0, 1, 2, ...) as-is
USE_LABELS = True

CHUNK_SIZE = 5_000_000  # number of edges to buffer before writing a parquet row group

print(f"Input size: {GML_PATH.stat().st_size / 1e9:.2f} GB")

## Step 1: Parse the file (nodes, then edges)

In GML all node blocks appear before the edge blocks, so a single pass over the file
first builds the id → label mapping and then streams the edges.

Depending on disk speed this can take several minutes.

In [ ]:
def convert_gml_to_parquet(gml_path, out_path, use_labels=True, chunk_size=5_000_000):
    labels = []          # index = internal id (networkx writes ids sequentially from 0)
    src_buf, tgt_buf = [], []
    n_edges = 0
    writer = None
    schema = pa.schema([("source", pa.int64()), ("target", pa.int64())])

    def flush():
        nonlocal writer, n_edges
        if not src_buf:
            return
        src = np.array(src_buf, dtype=np.int64)
        tgt = np.array(tgt_buf, dtype=np.int64)
        if use_labels:
            src = label_arr[src]
            tgt = label_arr[tgt]
        table = pa.table({"source": src, "target": tgt}, schema=schema)
        if writer is None:
            writer = pq.ParquetWriter(out_path, schema, compression="zstd")
        writer.write_table(table)
        n_edges += len(src_buf)
        src_buf.clear()
        tgt_buf.clear()

    t0 = time.time()
    label_arr = None
    in_edges = False

    with open(gml_path, "r", encoding="utf-8", buffering=1024 * 1024 * 8) as f:
        for line in f:
            s = line.strip()
            if not in_edges:
                if s.startswith("label"):
                    # label "288157"  →  288157
                    labels.append(int(s[7:-1]))
                elif s.startswith("source"):
                    # first edge reached: freeze the label list into a numpy array
                    label_arr = np.array(labels, dtype=np.int64)
                    labels = None
                    in_edges = True
                    print(f"Parsed {len(label_arr):,} nodes ({time.time()-t0:.0f}s). Parsing edges...")
                    src_buf.append(int(s[7:]))
            else:
                if s.startswith("source"):
                    src_buf.append(int(s[7:]))
                elif s.startswith("target"):
                    tgt_buf.append(int(s[7:]))
                    if len(tgt_buf) >= chunk_size:
                        flush()
                        print(f"  ... {n_edges:,} edges written ({time.time()-t0:.0f}s)")

    flush()
    if writer is not None:
        writer.close()
    print(f"Done: {len(label_arr):,} nodes, {n_edges:,} edges → {out_path.name} ({time.time()-t0:.0f}s)")
    return len(label_arr), n_edges


n_nodes, n_edges = convert_gml_to_parquet(GML_PATH, OUT_PATH, use_labels=USE_LABELS, chunk_size=CHUNK_SIZE)

## Step 2: Verify the output

In [ ]:
pf = pq.ParquetFile(OUT_PATH)
print(f"Output size : {OUT_PATH.stat().st_size / 1e6:,.1f} MB")
print(f"Rows (edges): {pf.metadata.num_rows:,}")
print(f"Schema      :\n{pf.schema_arrow}")

# Preview the first 10 edges
pf.read_row_group(0).slice(0, 10).to_pandas()

In [ ]:
# (Optional) Cross-check against the beginning of the GML file.
# The first edge in the GML is source 0 → target 1, and
# internal id 0 has label "1", id 1 has label "288157", so
# with USE_LABELS=True the first parquet row should be source=1, target=288157.
head = pf.read_row_group(0).slice(0, 1).to_pandas()
print(head)